This notebook has two ways of pre-processing the raw time series data:

1.   Filter and clean raw time series data

2.   Trial 1:
  *   Keep only completed rides with valid dropoffs
  *   Calculate and extract the time between a dropoff and a subsequent pickup for a particular geohash.

3.   Trial 2:
  *   Keep only pickups and offer date
  *   Ensure all pickups are valid



Step 1: Merging and setting up a basic data set from the raw time series data sets.

In [ ]:
#Ensure only these columns are in this dataset: Session_ID	Driver_ID	Ride_ID	Fare	Offer_Date	ETA	ETA_actual	End_Datetime	Ride_Time	Pickup_Geohash	Dropoff_Geohash	Current_Geohash	Time_to_Next_Ride
import pandas as pd
raw_data = "/pathto/raw_data.csv" # raw ehail time series data
raw_data_2 = "/pathto/raw_data_2.csv" # raw ETA data
rawdata_df = pd.read_csv(raw_data)
rawdata2_df = pd.read_csv(raw_data_2)

In [ ]:
mergedraw_df = pd.merge(rawdata_df, rawdata2_df, on=['SESSION_ID', 'MEDALLION_NUMBER', 'IMSI'], how='left')
completed_rides = mergedraw_df[(mergedraw_df['EHAIL_STATE'] == 'RIDER_DROPOFF') & (mergedraw_df['state'] == 'CONFIRMED')]

In [ ]:
import geohash2

coordinate_columns = [
    ('PIKCUP_LAT', 'PIKCUP_LON'),
    ('DROPOFF_LAT', 'DROPOFF_LON'),
    ('LAT','LON') ## Adjust columns for the function based on which columns need to be encoded
]

def encode_geohash(lat, lon, precision=6):
    return geohash2.encode(lat, lon, precision)

for lat_col, lon_col in coordinate_columns:
    geohash_col_name = f"{lat_col.split('_')[0]}_GEOHASH"
    completed_rides.loc[:,geohash_col_name] = completed_rides1.apply(lambda row:
                           encode_geohash(row[lat_col], row[lon_col], 6)
                           axis=1)

columns_to_drop = [col for pair in coordinate_columns for col in pair]
completed_rides.drop(columns=columns_to_drop, inplace=True)
completed_rides.head()

In [ ]:
# Filter out pre planned trips and remove trips with Taxibutler
temp = completed_rides[~((completed_rides1['DIRECT_DISPATCH'] == 1) & (completed_rides1['OFFER_TYPE'] != 'Uber'))]
filtered_df = temp[~((completed_rides1['OFFER_TYPE'] == 'TAXIBUTLER_SERVICE_F'))]

# Remove unused columns
columns_to_remove = [
    "DIRECT_DISPATCH", "MEDALLION_NUMBER",
    "IMSI", "TAXI_TYPE", "HACK_NUMBER",
    "state", "DISPATCH_SYSTEM_TYPE", "BATCH",
    "OFFER_TYPE", "EHAIL_STATE"
]
filter2df = filtered_df.drop(columns=columns_to_remove)
filter2df.head()

In [ ]:
# Rename columns
rename_dict = {
    'SESSION_ID': 'Session_ID',
    'PROVIDER_RIDE_ID': 'Ride_ID',
    'FLAT_FARE': 'Fare',
    'OFFER_DATE': 'Offer_Date',
    'MASTER_DRIVER_ID': 'Driver_ID',
    'ride_state_timestamp': 'End_Datetime',
    'actual_time_to_pickup': 'ETA_actual',
    'ride_time': 'Ride_Time',
    'PIKCUP_GEOHASH': 'Pickup_Geohash',
    'DROPOFF_GEOHASH': 'Dropoff_Geohash',
    'LAT_GEOHASH': 'Current_Geohash',
}

filter2df = filter2df.rename(columns=rename_dict)

#Re-order columns
new_order =[
    'Session_ID',
    'Driver_ID',
    'Ride_ID',
    'Fare',
    'Offer_Date',
    'ETA',
    'ETA_actual',
    'End_Datetime',
    'Ride_Time',
    'Pickup_Geohash',
    'Dropoff_Geohash',
    'Current_Geohash'
]
filter2df=filter2df[new_order]

In [ ]:
save_file_path = '/savepath/data.csv'
filter2df.to_csv(save_file_path, index=False)

Trial 1:

In [ ]:
## Optional Test, to merge geohash data by increasing the geohash radius for lower count geohashes
# Note: was not useful and was not used

geohash_counts = data['Dropoff_Geohash'].value_counts()

# Function to modify geohashes based on their count
def modify_geohash(geohash):
    count = geohash_counts[geohash]
    if count < 500:
        return geohash[:4]
    elif count < 1000:
        return geohash[:5]
    else:
        return geohash

# Apply the function to create a new column with modified geohashes
data['modified_geohash'] = data['Dropoff_Geohash'].apply(modify_geohash)

geohash_dict = {geohash: modify_geohash(geohash) for geohash in geohash_counts.index}

In [ ]:
#Cleanup
data = pd.read_csv(raw_data)
valid_eta_data = data[(data['ETA'] != 0) & (~data['ETA'].isna()) & (~data['ETA_actual'].isna())]
valid_eta_data['ETA_Ratio'] = valid_eta_data['ETA_actual'] / valid_eta_data['ETA']
average_eta_ratio = valid_eta_data['ETA_Ratio'].mean()
data['ETA_actual'] = data.apply(
    lambda row: row['ETA'] * average_eta_ratio if pd.isna(row['ETA_actual']) else row['ETA_actual'], axis=1
)
print(f'Missing ETA_actual values: {data["ETA_actual"].isna().sum()}')
data = data.dropna(subset=['Fare'])
data = data[data['Dropoff_Geohash'] != '000000']
data = data[data['Dropoff_Geohash'] != '7zzzzz']

In [ ]:
## Calculate time to next ride using subsequent picksups by merging dropoffs and pickups
# Create separate dataframes for dropoffs and pickups
dropoffs = data[['Ride_ID','End_Datetime', 'Dropoff_Geohash', 'Pickup_Geohash', 'Fare', 'ETA', 'ETA_actual']].copy()
pickups = data[['Offer_Date', 'Pickup_Geohash', 'Dropoff_Geohash']].copy()
rename_dict = {
    'Offer_Date': 'Offer_Date',
    'Dropoff_Geohash': 'Pickup_Geohash',
    'Pickup_Geohash': 'Dropoff_Geohash',
}

pickups = pickups.rename(columns=rename_dict)
# Merge the dropoffs with subsequent pickups within the same Dropoff_Geohash
merged_data = pd.merge_asof(
    dropoffs.sort_values('End_Datetime'),
    pickups.sort_values('Offer_Date'),
    left_on='End_Datetime',
    right_on='Offer_Date',
    by='Dropoff_Geohash',
    direction='forward',
    tolerance=pd.Timedelta('1H')
)

# Filter out rows where the pickup did not occur after the dropoff
merged_data = merged_data[merged_data['End_Datetime'] < merged_data['Offer_Date']]

# Calculate the time difference between the dropoff and the next pickup
merged_data['Time_to_Next_Ride'] = (merged_data['Offer_Date'] - merged_data['End_Datetime']).dt.total_seconds()
to_remove =['Pickup_Geohash_x',
            'Pickup_Geohash_y',
            'ETA',
            'ETA_actual',
            'Offer_Date',
            'Fare'
            ]

merged_data = merged_data.drop(columns=to_remove)

In [ ]:
## Add holidays
from pandas.tseries.holiday import USFederalHolidayCalendar

df['End_Datetime'] = pd.to_datetime(df['End_Datetime'])

# Add Time of Day column in decimal format
df['Time_of_Day'] = df['End_Datetime'].dt.hour + df['End_Datetime'].dt.minute / 60

# Add Day of Week column
df['Day_of_Week'] = df['End_Datetime'].dt.dayofweek  # Monday=0, Sunday=6

# Add Holiday column
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=df['End_Datetime'].min(), end=df['End_Datetime'].max())
df['Holiday'] = df['End_Datetime'].dt.date.isin(holidays.date).astype(int)
df =df.drop(['End_Datetime'], axis=1)

Trial 2: Only keeping offer date and pickup geohash

In [ ]:
raw_data = "/pathto/raw_data.csv"
df = pd.read_csv(raw_data)

In [ ]:
##
df_cleaned = df.dropna(subset=['PIKCUP_LAT', 'PIKCUP_LON'])
df_cleaned = df_cleaned.drop_duplicates(subset=['PROVIDER_RIDE_ID'])
duplicate_check = df_cleaned[df_cleaned.duplicated(subset=['PROVIDER_RIDE_ID'], keep=False)]
base_data = df_cleaned.copy()
base_data = base_data[['OFFER_DATE','PIKCUP_LAT','PIKCUP_LON']]

In [ ]:
coordinate_columns = [
    ('PIKCUP_LAT', 'PIKCUP_LON'),
]

def encode_geohash(lat, lon, precision=6):
    return geohash2.encode(lat, lon, precision)

for lat_col, lon_col in coordinate_columns:
    geohash_col_name = f"{lat_col.split('_')[0]}_GEOHASH"
    base_data.loc[:,geohash_col_name] = base_data.apply(lambda row:
                           encode_geohash(row[lat_col], row[lon_col], 6),
                           axis=1)

columns_to_drop = [col for pair in coordinate_columns for col in pair]
base_data.drop(columns=columns_to_drop, inplace=True)
base_data.head()

Rest of the code for preprocessing data is on the 'LowerManhattanFull' notebook